# Projet 9 - Mise en place d'un système RAG

#### Liste des imports du projet

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)   # affiche toutes les colonnes
pd.set_option('display.max_rows', 100)        # affiche jusqu'à 100 lignes (ajuste selon besoin)
pd.set_option('display.width', None)          # n'impose pas de largeur max, s'adapte au contenu
pd.set_option('display.max_colwidth', None)

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import torch
import langchain
import mistralai
import os

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from mistralai.client import Mistral

/var/folders/8p/5lplb74s1yxb9pvjkwtwxw_w0000gn/T/ipykernel_97629/4129900634.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


#### Importation des données du projet

Ici, nous avons sélectionné les événements culturels ayant eu lieu en bretagne sur toute la période 2026 et s'étalant jusqu'à 2027. Les données ont été exportées manuellement directement sur l'API OpenAgenda au format JSON.

In [2]:
from pathlib import Path

ROOT = Path.cwd().parent
DATA_FILE = ROOT/"data"
DATA = DATA_FILE / "data.json"

df = pd.read_json(DATA)

In [3]:
#Visualisation rapide de la composition du dataframe obtenue.
display(df)
print(df.shape)

,uid,slug,canonicalurl,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,image,imagecredits,thumbnail,originalimage,updatedat,daterange_fr,firstdate_begin,firstdate_end,lastdate_begin,lastdate_end,timings,accessibility,accessibility_label_fr,location_uid,location_coordinates,location_name,location_address,location_district,location_insee,location_postalcode,location_city,location_department,location_region,location_countrycode,location_image,location_imagecredits,location_phone,location_website,location_links,location_tags,location_description_fr,location_access_fr,attendancemode,onlineaccesslink,status,age_min,age_max,originagenda_title,originagenda_uid,contributor_email,contributor_contactnumber,contributor_contactname,contributor_contactposition,contributor_organization,category,country_fr,registration,links
0,42512026,le-voyage-dans-la-lune-marcel-dzama-7015150,https://openagenda.com/sortir-rennesmetropole/events/le-voyage-dans-la-lune-marcel-dzama-7015150,"« Le voyage dans la Lune », Marcel Dzama","« Le voyage dans la Lune » présente l’univers de l’artiste canadien Marcel Dzama à La Criée. Films, dessins et sculptures rendent hommage à Méliès et dévoilent un monde poétique, fantasque et politiq…","<p>La Criée présente du 14 février au 10 mai 2026 une exposition de Marcel Dzama, artiste canadien basé à New York, internationalement reconnu, mais rarement exposé en France. Intitulée « Le voyage dans la Lune », en clin d’œil au magicien des débuts du cinéma Georges Méliès, elle présente pour la première fois en Europe une large sélection des films de l’artiste, auxquels font écho un ensemble de storyboards, dessins, maquettes et costumes.</p>\n<p>Formé aux beaux-arts de Winnipeg, sa ville natale, immergé dès ses débuts dans les cultures populaires alternatives (il a notamment fait partie de groupes de rock et produit de nombreux fanzines), Marcel Dzama développe depuis la fin des années 1990 une œuvre foisonnante et joyeuse, au sein de laquelle la pratique du dessin et du film sont centrales.<br>Grand admirateur et fin connaisseur des débuts du cinéma (auquel nombre de ses films empruntent notamment le noir et blanc et la gestuelle expressionniste), Marcel Dzama est, plus largement, curieux d’univers variés : l’esprit surréaliste, les débuts du modernisme, la pop culture, l’illustration, la musique underground, etc. Ces références, qu’on reconnaîtra ou pas, nourrissent un univers de fantaisie – immédiatement reconnaissable lui -, tantôt merveilleux, tantôt cruel, tantôt poétique, tantôt politique.</p>\n<p>L’exposition « Le voyage dans la Lune » propose une quinzaine de films. À travers un programme d’une heure, imaginé par l’artiste, on découvre des films de jeunesse, des films tournés sur le vif, d’autres avec son fils et son père, d’autres encore produits dans le cadre de commandes. L’humour et l’inventivité irriguent d’une même énergie cette sélection. Marcel Dzama a par ailleurs choisi de mettre en avant deux films, présentés dans deux salles dédiées : « Une danse des bouffons » (2013) et « To live on the Moon (for lorca) » (2023). Le premier est inspiré par l’histoire d’amour entre Marcel Duchamp et la sculptrice Maria Martins, le second par les figures de la lune et du poète Frederico Garcia Lorca.</p>\n<p>En rebond aux films, des objets, sculptures, masques, dessins, maquettes, storyboard dessinés sont présentés dans l’espace central du centre d’art. Qu’ils s’agissent de costumes, d’éléments de décor ou de carnets dessins directement issus des films ou de dessins réalisés par ailleurs, on y retrouve, en couleur et en trait, les mêmes inspirations et la même facétie.</p>\n<p>Du crayon à la caméra, du poétique au politique, Marcel Dzama joue avec les codes et l’histoire du burlesque et du fantastique, avec une virtuosité de funambule. Ses œuvres mêlent références et pirouettes virtuoses avec un plaisir, une liberté et une joie qu’on espère contagieuses.</p>",Gratuit,[Exposition],https://cdn.openagenda.

(2424, 56)


## Phases d'exploration des données et de nettoyage

In [4]:
# Compréhension du dataframe
print("-"*100)
print("Liste des colonnes")
print(list(df.columns))

----------------------------------------------------------------------------------------------------
Liste des colonnes
['uid', 'slug', 'canonicalurl', 'title_fr', 'description_fr', 'longdescription_fr', 'conditions_fr', 'keywords_fr', 'image', 'imagecredits', 'thumbnail', 'originalimage', 'updatedat', 'daterange_fr', 'firstdate_begin', 'firstdate_end', 'lastdate_begin', 'lastdate_end', 'timings', 'accessibility', 'accessibility_label_fr', 'location_uid', 'location_coordinates', 'location_name', 'location_address', 'location_district', 'location_insee', 'location_postalcode', 'location_city', 'location_department', 'location_region', 'location_countrycode', 'location_image', 'location_imagecredits', 'location_phone', 'location_website', 'location_links', 'location_tags', 'location_description_fr', 'location_access_fr', 'attendancemode', 'onlineaccesslink', 'status', 'age_min', 'age_max', 'originagenda_title', 'originagenda_uid', 'contributor_email', 'contributor_contactnumber', 'contri

| Colonne | Catégorie | Rôle | Verdict |
|---|---|---|---|
| uid | Identification | Identifiant unique de l'événement | Conserver — clé primaire, citation |
| slug | Identification | Identifiant texte lisible (url) | Conserver |
| canonicalurl | Identification | URL de la fiche événement | Conserver — lien source |
| originagenda_title | Identification | Nom de l'agenda ayant publié l'événement | Conserver |
| originagenda_uid | Identification | Id de l'agenda source | Conserver — utile pour détecter les doublons de syndication |
| status | Identification | Statut (programmé / reporté / annulé) | Conserver — à utiliser comme filtre qualité |
| title_fr | Texte événement | Titre | Conserver — texte à vectoriser |
| description_fr | Texte événement | Description courte | Conserver — texte à vectoriser |
| longdescription_fr | Texte événement | Description longue | Conserver — texte à vectoriser (fallback si vide) |
| conditions_fr | Texte événement | Conditions, tarifs | Conserver — répond à "gratuit ?", "sur inscription ?" |
| keywords_fr | Texte événement | Mots-clés / catégories | Conserver — base du filtre culturel |
| daterange_fr | Dates | Plage de dates déjà formatée en texte | Conserver |
| firstdate_begin | Dates | Date début 1ère occurrence | Conserver — filtrage temporel |
| firstdate_end | Dates | Date fin 1ère occurrence | Conserver |
| lastdate_begin | Dates | Date début dernière occurrence | Conserver |
| lastdate_end | Dates | Date fin dernière occurrence | Conserver |
| timings | Dates | Détail des créneaux horaires par occurrence | Optionnel — utile si horaires précis affichés, sinon dropper |
| updatedat | Dates | Date de dernière modification de la fiche | Conserver (léger) — traçabilité, rafraîchissement futur |
| image | Images | URL image événement | Dropper — pas d'UI visuelle prévue |
| imagecredits | Images | Crédit photo événement | Dropper |
| thumbnail | Images | Vignette | Dropper |
| originalimage | Images | Image source | Dropper |
| location_image | Images | Image du lieu | Dropper |
| location_imagecredits | Images | Crédit photo du lieu | Dropper |
| location_uid | Localisation | Identifiant du lieu | Conserver |
| location_coordinates | Localisation | Latitude/longitude | Conserver — utile pour une future recherche de proximité |
| location_name | Localisation | Nom du lieu | Conserver — texte |
| location_address | Localisation | Adresse | Conserver |
| location_postalcode | Localisation | Code postal | Conserver |
| location_city | Localisation | Ville | Conserver — cœur du filtre géographique |
| location_department | Localisation | Département | Conserver |
| location_region | Localisation | Région | Conserver |
| location_countrycode | Localisation | Code pays | Dropper — constant sur ton sous-ensemble |
| country_fr | Localisation | Pays (texte) | Dropper — constant |
| location_district | Localisation | Quartier | Dropper — peu pertinent hors grandes métropoles |
| location_insee | Localisation | Code INSEE commune | Dropper — hors scope POC |
| location_phone | Localisation | Téléphone du lieu | Conserver |
| location_website | Localisation | Site web du lieu | Conserver |
| location_links | Localisation | Liens additionnels du lieu | Dropper — sparse et secondaire |
| location_tags | Localisation | Tags du lieu | Dropper — sparse et redondant avec keywords_fr |
| location_description_fr | Localisation | Description du lieu | Dropper — sparse, secondaire pour le POC |
| location_access_fr | Localisation | Accès au lieu (transport...) | Conserver |
| attendancemode | Modalités / public | Présentiel / en ligne / hybride | Conserver |
| onlineaccesslink | Modalités / public | Lien si événement en ligne | Conserver |
| accessibility | Modalités / public | Accessibilité handicap (code) | Conserver malgré la sparsité — vraie valeur quand renseigné |
| accessibility_label_fr | Modalités / public | Accessibilité handicap (libellé) | Conserver malgré la sparsité |
| age_min | Modalités / public | Âge minimum ciblé | Conserver malgré la sparsité |
| age_max | Modalités / public | Âge maximum ciblé | Conserver malgré la sparsité |
| registration | Modalités / public | Modalités d'inscription | Conserver |
| links | Modalités / public | Liens additionnels (billetterie...) | Dropper — sparse, secondaire pour le POC |
| contributor_email | Admin / technique | Email du contributeur | Dropper — vide à 100 %, et personnel si renseigné |
| contributor_contactnumber | Admin / technique | Téléphone du contributeur | Dropper — vide à 100 % |
| contributor_contactname | Admin / technique | Nom du contributeur | Dropper — vide à 100 % |
| contributor_contactposition | Admin / technique | Poste du contributeur | Dropper — vide à 100 % |
| contributor_organization | Admin / technique | Organisation du contributeur | Dropper — vide à 100 % |
| category | Admin / technique | Catégorie | Dropper — vide à 100 % |

### Nettoyage des colonnes / events avec statut complet, annulé ou reporté / events avec keywords non culturels

In [5]:
# Colonnes écartées d'après le tableau ci-dessus (hors-scope pour le RAG,
# ou entièrement vides). "_occurrence" est une colonne technique créée plus
# bas, ajoutée ici pour qu'elle soit nettoyée avec le reste au même endroit.
list_col_drop = [
    "contributor_organization", "category", "contributor_contactposition",
    "contributor_contactname", "contributor_contactnumber", "contributor_email",
    "links", "location_insee", "location_countrycode", "location_imagecredits",
    "location_image", "originalimage", "thumbnail", "imagecredits", "image",
    "_occurrence",
]

In [ ]:
print("-"*100)
print("Nombre de cellules null par variable")
print(df.isnull().sum())

print("-"*100)
print("Nombre de duplicats")
print(df['uid'].duplicated().sum())
print(df["slug"].duplicated().sum())
print(df["canonicalurl"].duplicated().sum())
print(df["title_fr"].duplicated().sum())
print(df["description_fr"].duplicated().sum())

----------------------------------------------------------------------------------------------------
Nombre de cellules null par colonne
uid                               0
slug                              0
canonicalurl                      0
title_fr                          0
description_fr                    0
longdescription_fr              111
conditions_fr                   674
keywords_fr                       0
image                           120
imagecredits                   1079
thumbnail                       120
originalimage                   120
updatedat                         0
daterange_fr                      0
firstdate_begin                   0
firstdate_end                     0
lastdate_begin                    0
lastdate_end                      0
timings                           0
accessibility                  1806
accessibility_label_fr         1806
location_uid                      0
location_coordinates              0
location_name                     0

On a observé qu'il y avait des duplicats au niveau du titre de l'événement, et de la description. Nous allons vérifié de quoi il s'agit.

In [7]:
dupes = df[df.duplicated(subset="title_fr", keep=False)].sort_values("title_fr")
dupes[["title_fr", "location_name", "firstdate_begin", "originagenda_uid", "uid"]]

,title_fr,location_name,firstdate_begin,originagenda_uid,uid
1290,"""Aux Abords de la Ville"" - Peintures et dessins de Sylvie Beaufils",La Cale,2026-05-17T15:00:00+02:00,38977657,1277727
897,"""Aux Abords de la Ville"" - Peintures et dessins de Sylvie Beaufils",La Cale,2026-04-19T15:00:00+02:00,38977657,12930153
1042,"""Aux Abords de la Ville"" - Peintures et dessins de Sylvie Beaufils",La Cale,2026-04-26T15:00:00+02:00,38977657,52008733
2190,"""Aux Abords de la Ville"" - Peintures et dessins de Sylvie Beaufils",La Cale,2026-05-03T15:00:00+02:00,38977657,71465843
215,"""Aux Abords de la Ville"" - Peintures et dessins de Sylvie Beaufils",La Cale,2026-05-10T15:00:00+02:00,38977657,77470732
...,...,...,...,...,...
597,À VOS AMOURS • ANNE-CÉCILE ESTEVE (FR),Quartier La Courrouze,2025-12-21T09:00:00+01:00,38977657,93447350
428,À VOS AMOURS • ANNE-CÉCILE ESTEVE (FR),Quartier La Courrouze,2025-11-30T09:00:00+01:00,38977657,51573645
596,À VOS AMOURS • ANNE-CÉCILE ESTEVE (FR),Quartier La Courrouze,2025-12-14T09:00:00+01:00,38977657,87990907
1568,À VOS AMOURS • ANNE-CÉCILE ESTEVE (FR),Quartier La Courrouze,2025-12-28T09:00:00+01:00,38977657,5669703


Nous observons des duplications d'événements qui proviennent en réalité de dates multiples pour un même évenemment (simplement un événements répétés régulièrement). Il est important pour la mise en base vectorielle que ces événements soient compactés en un seul car sinon en cas de question du type "top 5 des événements dans tel domaine" on pourrait avoir 5 fois le meme evenement qui sortirait pour des dates différentes. Nous allons donc aggrégés les évenements un un seul mais avec une conservation de l'ensemble des dates sous forme de liste.

In [8]:
# Premier essai de regroupement, à titre exploratoire (df encore complet,
# statut pas encore filtré). La version définitive, df_evenements_clean,
# est plus bas, une fois le filtre statut appliqué.

# 1. df de départ : 56 colonnes, ~2424 lignes (1 ligne = 1 séance)

# 2. colonne technique pour coupler début/fin de la même séance
df["_occurrence"] = list(zip(df["firstdate_begin"], df["firstdate_end"]))

# 3. uniquement les colonnes calculées, indexées par (titre, lieu)
agg_dates = (
    df.groupby(["title_fr", "location_name"])
      .agg(
          uids=("uid", list),
          dates=("_occurrence", lambda s: sorted(set(s))),
          date_min=("firstdate_begin", "min"),
          date_max=("lastdate_end", "max"),
          nb_occurrences=("uid", "count"),
      )
)

# 4. toutes les colonnes originales, une ligne par groupe (occurrence la plus ancienne)
representative = (
    df.sort_values("firstdate_begin")
      .groupby(["title_fr", "location_name"])
      .first()
)

# 5. on recolle les deux sur leur index commun (title_fr, location_name)
df_evenements = representative.join(
    agg_dates[["uids", "dates", "date_min", "date_max", "nb_occurrences"]]
).reset_index()

print(df.shape)
print(df_evenements.shape)
print(list(df_evenements.columns))

(2424, 57)
(2038, 62)
['title_fr', 'location_name', 'uid', 'slug', 'canonicalurl', 'description_fr', 'longdescription_fr', 'conditions_fr', 'keywords_fr', 'image', 'imagecredits', 'thumbnail', 'originalimage', 'updatedat', 'daterange_fr', 'firstdate_begin', 'firstdate_end', 'lastdate_begin', 'lastdate_end', 'timings', 'accessibility', 'accessibility_label_fr', 'location_uid', 'location_coordinates', 'location_address', 'location_district', 'location_insee', 'location_postalcode', 'location_city', 'location_department', 'location_region', 'location_countrycode', 'location_image', 'location_imagecredits', 'location_phone', 'location_website', 'location_links', 'location_tags', 'location_description_fr', 'location_access_fr', 'attendancemode', 'onlineaccesslink', 'status', 'age_min', 'age_max', 'originagenda_title', 'originagenda_uid', 'contributor_email', 'contributor_contactnumber', 'contributor_contactname', 'contributor_contactposition', 'contributor_organization', 'category', 'countr

Maintenant que nous avons aggégés les événements, nous pouvons effectuer le nettoyage réel du tableau avec la suppression des colonnes vides ou non-pertinentes pour une demande client sur un événement.

Nous avons maintenant un df qui est nettoyé et qui contient seulement les informations pertinentes pour un utilisateur. Nous avons donc regardé au niveau général et allons maintenant descencdre d'un cran en nous intéréssant réellement aux evenements, ont ils eu lieux (annulations) ? Est il encore possible de s'y inscrire (complet ou non) ? En prenant en compte la variable statut. Nous regarderons ensuite si les keywords des événements correspondent bien à ce que nous avons sélectionné et que nous avons bel et bien listé des évenements culturels.

In [9]:
# Nettoyage du statut : la colonne `status` est une chaîne JSON (id + libellés
# multilingues), pas une catégorie directement exploitable. On la parse pour
# ne garder que les événements réellement programmés AVANT de dédupliquer,
# afin qu'aucune séance annulée ne se retrouve cachée dans les listes de dates
# des événements récurrents regroupés plus bas.

import json

def parse_status(s):
    try:
        d = json.loads(s)
        return d["id"], d["label"].get("fr")
    except (TypeError, ValueError, KeyError):
        return None, None

df[["status_id", "status_label"]] = df["status"].apply(lambda s: pd.Series(parse_status(s)))
print(df["status_label"].value_counts())

df = df[df["status_id"] == 1].copy()
print(df.shape)  # attendu : ~2410 lignes

# garde-fou : arrête le notebook immédiatement si un statut non "Programmé"
# a échappé au filtre, plutôt que de laisser l'erreur se propager en silence
assert df["status_id"].eq(1).all()

status_label
Programmé    2410
Annulé          8
Complet         5
Reporté         1
Name: count, dtype: int64
(2410, 59)


In [10]:
# df est maintenant le DataFrame filtré à l'étape précédente (status_id == 1)

df["_occurrence"] = list(zip(df["firstdate_begin"], df["firstdate_end"]))

agg_dates = (
    df.groupby(["title_fr", "location_name"])
      .agg(
          uids=("uid", list),
          dates=("_occurrence", lambda s: sorted(set(s))),
          date_min=("firstdate_begin", "min"),
          date_max=("lastdate_end", "max"),
          nb_occurrences=("uid", "count"),
      )
)

representative = df.sort_values("firstdate_begin").groupby(["title_fr", "location_name"]).first()

df_evenements_clean = (
    representative.join(agg_dates[["uids", "dates", "date_min", "date_max", "nb_occurrences"]])
                  .reset_index()
                  .drop(columns=list_col_drop, errors="ignore")
)

print(df_evenements_clean.shape)
print(df_evenements_clean["status_label"].value_counts() if "status_label" in df_evenements_clean else "pas de colonne status_label — normal, déjà filtré en amont")

(2027, 48)
status_label
Programmé    2027
Name: count, dtype: int64


Vérifions maintenant les keywords pour voir si potentiellement certains ne correspondent pas à ce qu'on attend.

In [11]:
#On déballe la liste "keywords_fr" pour chaque ligne du df et on comptabilise ce qui sort, on empêche aussi la casse en mettant tout en minuscule.
counts = df_evenements_clean["keywords_fr"].explode().dropna().str.lower().value_counts()
counts.head(100)

keywords_fr
théâtre                        390
musique                        336
spectacle                      262
atelier                        230
rennes                         224
festival                       216
exposition                     208
danse                          170
concert                        167
cinéma                         145
ciné manivel                    90
rencontre                       86
patrimoine                      76
conte                           68
famille                         67
art                             66
bretagne                        64
tout public                     62
gratuit                         61
animation                       60
café théâtre                    57
comédie                         57
cinema                          55
en physique                     54
culture                         52
humour                          51
documentaire                    50
enfants                         49
jeune pu

vérifier
liste_à_supp = ["détection de potentiel", "marché du travail", "s'informer", "découverte secteur / métier", ]

In [12]:
#Ici on a regardé tous les keywords pouvant être problématique pour voir à quel type d'événement ils correspondaient pour voir si on supprime ou pas.
df_evenements_clean[df_evenements_clean["keywords_fr"].apply(lambda l: "maison de quartier" in [k.lower() for k in l])][["title_fr","keywords_fr"]]

,title_fr,keywords_fr
54,APPEL À TISSUS !,"[ateliers, créatif, décoration, rennes, bellangerais, maison de quartier, fête]"
60,ATELIER CAFET’ : Impression 3D,"[atelier, impression 3D, maison de quartier, bellangerais, rennes, numérique responsable]"
65,ATELIER CRÉATION NUMÉRIQUE,"[open lab, rennes, maison de quartier, bellangerais, atelier]"
132,Atelier Création d’encres végétales,"[atelier, créatif, encres végétales, arts plastiques, maison de quartier, bellangerais]"
142,"Atelier Gravure sur Tetrapak ( ouvert à toutes et tous, même débutants !)","[atelier, gravure, rennes, maison de quartier, la bellangerais]"
145,Atelier Leadership,"[atelier, femme, rennes, maison de quartier, bellangerais]"
159,Atelier créatif numérique : impression 3D,"[rennes, impression 3d, maison de quartier, bellangerais, atelier]"
163,Atelier cuisine,"[atelier, cuisine, maison de quartier, bellangerais]"
164,Atelier cuisine - Goûter antigaspi,"[atelier, cuisine, antigaspi, rennes, bellangerais, maison de quartier]"
177,Atelier décoration Suprise party & Place au miam,"[atelier, création, bellangerais, maison de quartier, rennes]"


In [13]:
# On établit la liste des mots à supprimer dans les keywords pour exclure les lignes qui présentent ces keywords.
liste_a_supp = [
    "détection de potentiel", "marché du travail", "s'informer",
    "découverte secteur / métier", "opportunité d'emploi",
    "1 jeune  1 solution", "se préparer", "Aides à l'emploi", 
    "Création d'entreprise", "Emploi/emploi", "Insertion, entreprise", 
    "entreprises", "outil métier"
]

mask_a_exclure = df_evenements_clean["keywords_fr"].apply(
    lambda kws: any(k.lower() in liste_a_supp for k in kws)
)
df_evenements_final = df_evenements_clean[~mask_a_exclure].copy()

print(df_evenements_clean.shape, "->", df_evenements_final.shape)

(2027, 48) -> (1972, 48)


### Nettoyage du texte
Nous allons maintenant regarder le texte présent dans les différentes variables pour les nettoyer :
- balises html
- entités html
- espace et caractères invisibles
- encodage
- valeurs manquantes et texte insuffisant
- doublons internes au texte

Les colonnes textes d'intérets pour notre projet sont : "title_fr", "description_fr", "longdescription_fr", 
"conditions_fr", "location_name", "location_address", "location_access_fr".

In [14]:
import re
import html 
from bs4 import BeautifulSoup
from bs4 import MarkupResemblesLocatorWarning
import warnings

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

#On créer la liste des balises qui servent à structurer le html.

BALISES_BLOC = ["p", "br", "li", "div", "tr", "h1", "h2", "h3", "h4", "h5", "h6"]

#On créer une fonction qui supprime les cellules si il n'y a rien dedans et si on a du texte on le 
# parse avec le html.parser via Beautifoulsup4 puis on récupère ce texte via le html.unescape(soup.get_text) 
# qui remplace toutes les entités html par leur réel symbole et qui récupère le texte en séparant le texte de leures balises avec un espace
def nettoyer_html(texte):
    if not isinstance(texte, str) or not texte.strip():
        return ""
    soup = BeautifulSoup(texte, "html.parser")
    return html.unescape(soup.get_text(separator=" "))

#On normalise les espaces : Cette fonction nettoie et formate un texte pour qu'il ne reste que des espaces simples entre les mots.

def normaliser_espaces(texte):
    texte = texte.replace("\xa0", " ") #Remplace les espaces incassables (fréquents sur le Web ou dans Word) par un espace standard.
    texte = re.sub(r"[\u200b\ufeff]", "", texte) #Supprime les caractères invisibles masqués dans le texte (comme l'espace de largeur nulle \u200b ou le marqueur d'ordre des octets \ufeff).
    texte = re.sub(r"\s+", " ", texte) #re.sub(r"\s+", " ", texte) 
    return texte.strip() #Supprime les espaces restants tout au début et toute la fin du texte, puis renvoie le résultat.

#Ici la fonction orchestratrice des deux précédentes.
def nettoyer_texte(texte):
    return normaliser_espaces(nettoyer_html(texte))

#Colonnes qui seront à vectoriser.
colonnes_texte = [
    "title_fr", "description_fr", "longdescription_fr", "conditions_fr",
    "location_name", "location_address", "location_access_fr",
]

#Application de la fonction sur ces colonnes dans notre df d'intérêt.
for col in colonnes_texte :
    df_evenements_final[col] = df_evenements_final[col].apply(nettoyer_texte)

df_evenements_final["texte_description"] = df_evenements_final.apply(
    lambda row: row["longdescription_fr"] if len(row["longdescription_fr"]) > len(row["description_fr"])
    else row["description_fr"],
    axis=1
)

In [17]:
df_evenements_final[["texte_description"]].sample(5)

,texte_description
1479,"Les ateliers des arts plastiques s’exposent dans la galerie du Lavoir ! Découvrez les différentes techniques développées toute l’année (peinture, gravure, sculpture, collage…) et la créativité dont font preuve les participantes et participants des ateliers de François Soutif."
239,"MUSIQUE • TOUT PUBLIC • GRATUIT Découvrez ou redécouvrez la Block Party rennaise de Dooinit et rejoignez le parc des Hautes-Ourmes autour d’un barbecue, de jeux pour enfants, d’une animation breakdance avec la compagnie Primitif et d’une initiation au graffiti avec le Fresh Street Art. Des transats et un espace aménagé pour les amateurs et amatrices de danse seront mis à disposition. Côté musique, le producteur rennais J-Zen, signé sur le label Dooinit Music, mettra cette journée en musique avec son alliance de beats lourds et de samples triturés au point de les rendre méconnaissables… Ayã Brown Carvalho, aka DJ Flaya, puisera, quant à lui, dans son patrimoine culturel métissé entre le Brésil et l’Angleterre ainsi que dans son goût prononcé pour les musiques issues de la culture hip-hop, pour nous proposer un panel de musiques à la fois versatile et spécialisé. Un beau dimanche en perspective ! > DJ sets à partir de 15h • Renseignements sur dooinit-festival.com"
1510,"En trois tableaux et six personnages, ce spectacle de marionnettes aussi drôle que tragique nous propose tout simplement de rire de la fin du monde. Pour commencer, Bradi le paresseux et Toto le singe se questionnent. Y a-t-il un ailleurs où se rendre, en dehors de leur boîte ? Dans le deuxième opus, Crocuta Crocuta, la hyène tachetée et le lombric terrestre sentent la terre disparaître sous leurs pattes. Déployer des stratégies de survie semble voué à l’échec… Dans la dernière histoire, deux créatures squelettiques coincées sous terre perdent la mémoire jusqu’à oublier le nom de leur propre espèce. Comment vont-ils faire pour ne pas disparaitre une deuxième fois, irréversiblement ? Dans un dépouillement qui laisse voir tout le travail du marionnettiste, les deux manipulateurs animent un bestiaire attachant aux prises avec son environnement ; animaux et fossiles face à l’adversité, démunis devant la catastrophe à venir."
424,"« Magicien ? C’est pas un métier ! Heureusement que je n’ai pas écouté Marcelle, ma grand-mère... Sinon je ne pourrais pas vous raconter comment j’ai convaincu Chloé d’aller au cinéma avec moi à 14 ans, mes conneries avec Simon, mes expériences de G.O au Club Med ou encore mes premières scènes à Paris ! Et tout ça... grâce à la magie ! » En 2022, Clément co-écrit son premier spectacle d’humour et de magie avec Nicolas Genevaz et Thomas Caruso Aragona destiné au grand public et le joue un an à Paris au Théâtre des Mathurins et à l’Apollo. S’ensuit un premier festival d’Avignon complet et une tournée dans toute la France. il est également la révélation « Coup de coeur » dans l’émission « Les Comiques Préférés des Français » avec Laurence Boccolini, il fait une apparition dans « Cabaret » avec Stéphane Bern sur France 2, rejoint le casting de la saison 14 du Jamel Comedy Club sur Canal + et termine finaliste de la 20ème saison de « La France a un Incroyable Talent » sur M6. Trempé d’humour et de réparties, son premier spectacle raconte avec énergie sa vie de magicien, ses débuts, ses copines, ses conneries, son arrivée à Paris et ses expériences de G.O au Club Med ! __________ « Le coup de coeur de l’émission. Bluffant ! » - France 2. « Une révélation que la découverte de ce magicien charismatique ! » - Ouest Mag. « Celui qui était annoncé comme magicien est bien plus que cela ! Drôle, malicieux et un brin charmeur… » - L’Est Eclair. https://youtu.be/L_o4_7oQyT4"
271,"Info de dernière minute : le concert Qui parle ombre , qui devait réunir Gavin Bryars et le duo Midget ! à l’Antipode, est annulé. Ce dimanche sera tout de même l’occasion de fêter l’ouverture du festival Autres Mesures avec un concert de Ben Bertrand au Musée des Beaux-Arts ! C

### On vérifie maintenant que tous les événément est une description assez longue pour être assez informative ou au moins qui n'est pas une simple répétition du titre de l'événement.

In [18]:
df_evenements_final["texte_description"].str.len().describe()

count    1972.000000
mean      757.842292
std       605.072310
min         5.000000
25%       354.750000
50%       614.000000
75%       982.250000
max      7116.000000
Name: texte_description, dtype: float64

In [20]:
df_tmp = df_evenements_final.assign(len_txt=df_evenements_final["texte_description"].str.len())
df_tmp.nsmallest(30, "len_txt")[["title_fr", "texte_description", "len_txt"]]

,title_fr,texte_description,len_txt
1576,RUMBA,Rumba,5
730,FILM INATTENDU,Film inattendu,14
1625,SAVE OUR SOULS,Save our souls,14
340,CONFÉRENCE - AVC,Conférence - avc,16
67,AU COEUR DU BELON,Au coeur du belon,17
1144,LES AMES BOSSALES,Les ames bossales,17
1455,OPERA LA FLUTE ENCHANTEE,Opera la flute enchantee,24
39,AGUIRRE LA COLERE DE DIEU,Aguirre la colere de dieu,25
1074,L'EAU DANS TOUS SES ETATS,L'eau dans tous ses etats,25
1107,LA NOUVELLE AVENTURE MOBILE,La nouvelle aventure mobile,27


In [ ]:
seuil_min = 36 #On voit que c'est à partir de 36 caractère qu'on obtient des évenements avec un minimum d'information.
df_evenements_final = df_evenements_final[df_tmp["len_txt"] > seuil_min].copy()
print(df_evenements_final.shape)